# 🥈 notebook Silver

In [0]:
df_bronze = spark.table("bronze_walmart_sales")

display(df_bronze)

## 🧼 Limpeza + tipagem:

In [0]:
from pyspark.sql.functions import to_date, col

df_clean = (
    df_bronze
    .withColumn("Date", to_date(col("Date"), "dd-MM-yyyy"))
    .filter(col("Store").isNotNull())
)

## 🧱 Criar tabela Silver:

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_walmart_sales
USING DELTA
AS SELECT * FROM bronze_walmart_sales WHERE 1=0
""")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "silver_walmart_sales")

(
    delta_table.alias("target")
    .merge(
        df_clean.alias("source"),
        "target.Store = source.Store AND target.Date = source.Date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.sql("SELECT * FROM silver_walmart_sales").show()

In [0]:
df_new = df_clean.filter(col("Date") > "2012-01-01") \
                 .dropDuplicates(["Store", "Date"])
(
    delta_table.alias("target")
    .merge(
        df_new.alias("source"),
        "target.Store = source.Store AND target.Date = source.Date"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.sql("""
SELECT Store, COUNT(*) as total
FROM silver_walmart_sales
GROUP BY Store
ORDER BY total DESC
""").show()

In [0]:
spark.sql("SELECT COUNT(*) FROM silver_walmart_sales").show()